# Exercise 2 – Exploring Unlabelled MMS Ion Spectra

**Companion notebook:** `02_data_exploration.ipynb`

**Companion dataset:** `data/ex1_cleaned_unlabelled.nc` (produced in Exercise 1)

Work through the cells in order. Each exercise has a **Problem** statement in markdown followed by a code cell with `# YOUR CODE HERE`. The patterns you need are all in the companion notebook — re-read it if you get stuck.


In [1]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.decomposition import PCA
from spacephyml.datasets.mms import SpectrumDataset

sns.set_theme()
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

figure_path = Path('./figures')
figure_path.mkdir(exist_ok=True)

RANDOM_STATE = 42
NC_FILE = 'data/ex1_cleaned_unlabelled.nc'   # produced in Exercise 1

---

## Background

In the companion notebook you explored a labelled MMS dataset: every 5-minute window carried a ground-truth region label (Solar Wind, Ion Foreshock, Magnetosheath, Magnetosphere), and you could colour every plot by that label to see whether the data made physical sense.

Here you only have the cleaned but **unlabelled** dataset from Exercise 1 (March 2018). The spacecraft was still flying through the same regions — it just did not tell you which one. This is the situation you face whenever you want to apply a trained model to new data, or when you are exploring new data for which no annotations exist yet.

Without labels you cannot check class balance, and you cannot colour-code a PCA scatter plot by region. Instead you will:

- Visualise spectra and try to recognise familiar patterns by eye.
- Use PCA purely as an *exploration* tool, colouring points by time or by the value of a derived scalar (e.g. total flux) instead of by region.
- Look for structure (clusters, gradients, outliers) in PC space and reason about what physical populations might explain it.

This workflow — *structure first, interpretation second* — is the basis of all unsupervised analysis in space physics.

---

### Exercise 2.1 – Load the cleaned dataset

Load `NC_FILE` into a `SpectrumDataset` with a 5-minute window length (`N = 66`) and `flatten=False` so each sample keeps its 2-D shape `(N, n_bins)`. Do **not** pass a label column — there is none.

Print the total number of windows and the shape of one sample.

In [ ]:
N = 133   # ≈ 10 minutes at 4.5 s cadence

# YOUR CODE HERE
# 1. Create a SpectrumDataset from NC_FILE with N=133, flatten=False.
# 2. Print the total number of windows.
# 3. Print the shape of ds[0][0] (the first sample's data array).
#
# Tip: SpectrumDataset returns (data, label) tuples. When no label column
# is present the label tensor will be a placeholder — ignore it here.


### Exercise 2.2 – Visualise a sample of spectra

Without labels, you cannot sort windows by region. Instead, pick **six** windows spread evenly across the dataset (e.g. indices 0, N//5, 2*N//5, …) and plot each as a spectrogram.

Use `utils.plot_functions.spectrum_plot` as in the companion notebook. Label each panel with the window index and the approximate start time (retrieve it from `ds.timestamps` if available, or just use the index).

**After plotting, write a short comment below each subplot** (as a string in a `ax.text` or `ax.set_title` call) noting whether it looks more like Solar Wind, Magnetosheath, or Magnetosphere, based on the spectral fingerprints from the companion notebook Background section.

In [ ]:
from utils.plot_functions import spectrum_plot

spec_lev = ds.bin_centers[0, 0]   # energy bin centres, shape (n_bins,)
X_all, _ = ds[:]                  # retrieve all windows

n_show = 6
show_idxs = np.linspace(0, len(ds) - 1, n_show, dtype=int)
zlim = [20, 1.5e08]

# YOUR CODE HERE
# Create a figure with 2 rows × 3 columns.
# For each of the 6 indices in show_idxs:
#   1. Call spectrum_plot to render the spectrogram in the appropriate axis.
#   2. Set the title to something like f'Window {idx}  →  [your guess]'
#      where [your guess] is the region you think this window belongs to.
# Save the figure to figure_path / '02a_sample_spectra.png'.


---

### Exercise 2.3 – A note on dynamic range and preprocessing

Before running PCA on raw flux values, we need to compress the dynamic range. The companion notebook explains why: raw fluxes span many orders of magnitude, which would let a handful of high-flux observations dominate the principal components.

Copy the `preprocess` function from the companion notebook into the cell below. Then apply it to the full dataset using `'log10p1'` scaling and confirm that the output values fall in a sensible range by printing the global min, max, and mean.

In [ ]:
# YOUR CODE HERE
# 1. Define the preprocess(X, scaling) function (copy from companion notebook).
# 2. Load all windows with flatten=True and apply log10p1 scaling.
# 3. Print global min, max, and mean of the scaled array.


---

### Exercise 2.4 – Scree plot for the unlabelled dataset

Fit a PCA model with `n_components=20` on the full scaled dataset (use `log10p1` scaling and `N=133`).

Plot a **scree plot** (explained variance ratio per component) and a **cumulative explained variance** curve on the same figure. Add a horizontal dashed line at 60 % and annotate which component first crosses that threshold.

Does the result look similar to the labelled dataset from the companion notebook? What does the shape of the scree curve tell you about the intrinsic dimensionality of MMS ion spectra?

In [ ]:
N_COMP = 20

# YOUR CODE HERE
# 1. Load the full dataset with flatten=True, N=133.
# 2. Apply log10p1 preprocessing.
# 3. Fit PCA(n_components=N_COMP, random_state=RANDOM_STATE).
# 4. Plot explained variance ratio (bar chart, left y-axis) and
#    cumulative explained variance (line, right y-axis) on the same axes.
#    Add a horizontal dashed line at 0.90 and annotate the crossing point.
# 5. Save to figure_path / '02a_scree_plot.png'.


---

### Exercise 2.5 – PC scatter plot coloured by time

Because we have no labels, we use **time** as the colour axis instead of region. Project every window onto its first two principal components (PC1 and PC2) and make a scatter plot where each point is coloured by its position in the month (i.e. its window index, mapped to a sequential colormap such as `'plasma'`).

Add a colourbar labelled `'Window index (time →)'`.

1. Do points from early in the month cluster together, or are they scattered throughout PC space?
2. Are there any obvious gaps or dense clumps in the scatter? Where would you draw boundary lines if you had to separate the data into two or three groups by eye?

In [ ]:
# YOUR CODE HERE
# 1. Project all windows onto PC1 and PC2 using the PCA fitted in 2.4.
#    (Hint: pca.transform(X_scaled))
# 2. Create a scatter plot coloured by window index.
#    Use plt.scatter with c=np.arange(len(X_scaled)) and a sequential cmap.
# 3. Label axes 'PC1 (explained var: X%)' and 'PC2 (explained var: Y%)'.
# 4. Add a colourbar.
# 5. Save to figure_path / '02a_pca_time.png'.


---

### Exercise 2.6 – PC scatter plot coloured by a physical proxy

Plasma regions differ in their total ion flux as well as their spectral shape. A simple scalar proxy for the overall energy content is the **log mean flux** across all 32 energy bins:

$$\bar{F} = \frac{1}{N \cdot n_{\text{bins}}} \sum_{t,\,e} \log_{10}(1 + x_{t,e})$$

Compute this for every window and use it as the colour axis in a new PC1–PC2 scatter plot. Also make a PC1–PC3 scatter.

Arrange the two plots side-by-side in one figure. Compare the colour gradient to the orbit plot from Exercise 1.2b: do points with high mean flux correspond to time periods when the spacecraft was likely outside the magnetosphere?

> **Hint:** You can compute the mean flux before or after flattening — the result is the same. Use the scaled (log10p1) matrix.

In [ ]:
# YOUR CODE HERE
# 1. Compute log_mean_flux for each window (shape: (n_windows,)).
# 2. Make a 1×2 figure:
#    - Left panel: PC1 vs PC2, coloured by log_mean_flux.
#    - Right panel: PC1 vs PC3, coloured by log_mean_flux.
#    Use a diverging or sequential colormap that highlights high vs low values.
# 3. Add a shared colourbar labelled 'Mean log₁₀(1+flux)'.
# 4. Save to figure_path / '02a_pca_flux.png'.


---

### Exercise 2.7 – Window-length sensitivity (without labels)

The companion notebook showed how the PCA sensitivity to window length can be evaluated. Repeat the same sweep here — window lengths of 1, 5, 10, and 20 minutes — but this time **only** for `log10p1` scaling (no comparison needed). For each window length:

1. Load up to 300 windows from the unlabelled dataset.
2. Apply log10p1 preprocessing.
3. Fit `PCA(n_components=20)` and record the explained-variance ratios.

Plot a scree plot that overlays all four window lengths on the same axes (different colours, labelled in the legend). Does the unlabelled dataset show the same trend as the labelled one in the companion notebook?

In [ ]:
N_COMP      = 20
N_minutes   = np.array([1, 5, 10, 20], dtype=np.int64)
N_steps     = (N_minutes * 60 / 4.5).astype(np.int64)
max_samples = 300   # same cap for all window lengths

# YOUR CODE HERE
# For each entry in N_steps:
#   1. Load the dataset with that N and samples=max_samples, flatten=True.
#   2. Apply log10p1 preprocessing.
#   3. Fit PCA(n_components=N_COMP) and store evr (explained_variance_ratio_).
#
# Then plot all four scree curves on the same axes, one colour per window
# length, with a legend like '1 min', '5 min', etc.
# Save to figure_path / '02a_scree_windows.png'.


---

### Exercise 2.8 – Principal component vectors as spectrograms

Each principal component is a weight vector of length `N × n_bins`. Reshaping it back to `(N, n_bins)` lets us display it as a spectrogram — directly comparable to the raw data in Exercise 2.2.

Using the PCA fitted in Exercise 2.4 (10-minute windows, `log10p1` scaling), plot the first **three** principal components as spectrograms. Use a **diverging** colourmap (e.g. `'RdBu_r'`) centred at zero: red = positive loading (high flux here *increases* the projection), blue = negative.

Below each panel write a one-sentence interpretation: what spectral feature does this component appear to capture? Compare your interpretation to the companion notebook's discussion of PC1, PC2, and PC3.

In [ ]:
N_PCS_VIZ = 3

spec_lev  = ds.bin_centers[0, 0]     # energy bin centres (n_bins,)
n_bins    = len(spec_lev)
time_axis = np.arange(N)

# YOUR CODE HERE
# 1. Retrieve pca.components_  (shape: N_COMP × (N*n_bins)).
# 2. For each of the first N_PCS_VIZ components:
#    a. Reshape the vector to (N, n_bins).
#    b. Display it with plt.pcolormesh using a diverging cmap and
#       vmin=-|max|, vmax=|max| so the scale is symmetric around zero.
#    c. Set y-axis to log scale (plt.yscale('log')) and label it 'Energy (eV)'.
#    d. Set x-axis label 'Time step'.
#    e. Add a colourbar and a title 'PC{i+1}'.
# 3. Save to figure_path / '02a_pc_spectrograms.png'.


---

### Exercise 2.9 – Visual clustering: can you identify plasma regions? *(Reflection)*

Look back at the PC1–PC2 scatter plots you made in Exercises 2.5 and 2.6. In the labelled companion notebook the four regions were clearly separated along the first two PCs.

Answer the following questions in the markdown cell below (edit it directly):

1. Without labels, can you still see evidence of distinct groups in PC space? Describe what you observe (e.g. elongated clouds, isolated clumps, a continuous gradient).
2. The log mean flux proxy (Exercise 2.6) acted as a stand-in for a region label. How well did it separate the data? What are its limitations compared to a true label?
3. Suppose you were going to apply a k-means or Gaussian mixture model to find clusters automatically. Based on what you see in the scatter plots, how many clusters would you try first, and why?

*(Double-click to edit — write your answers here)*

1. 
2. 
3. 

---

## Summary

In this exercise you applied the data-exploration pipeline from the companion notebook to an **unlabelled** MMS dataset:

- Ion spectrograms still show recognisable fingerprints — narrow high-energy peaks (solar wind / foreshock) and broad medium-energy bumps (magnetosheath) — even without ground-truth labels. Visual inspection remains a powerful first check.
- PCA reduces the 32×66-dimensional spectra to a handful of dominant modes. The scree curve and the 60%-variance threshold are properties of the data structure, not of the labels, so they apply equally well to unlabelled datasets.
- Without labels you colour PCA scatter plots by time or by a physical proxy (log mean flux). Structure is still visible — the proxy separates parts of PC space — but it cannot replace true region labels for quantitative analysis.

**Key principle:** PCA reveals structure in the data regardless of labels. Labels help you *name* that structure — but looking for it first, without assumptions, is good scientific practice.

### Further reading
- Toy-Edens et al. (2024) — unsupervised classification of 8 years of MMS dayside regions: https://doi.org/10.1029/2024JA032431
- Olshevsky et al. — labelled MMS dataset: https://doi.org/10.5281/zenodo.17152371
